# Solution B3 · Tuning XGBoost with a random search (and what it is worth)

**Type**: Implement with scikit-learn

## Objective

Search the XGBoost hyperparameters with `RandomizedSearchCV` (20 combinations over `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `reg_lambda`, `gamma`) in a 3-fold cross-validation, then fix the rounds with **early stopping** on a validation set and evaluate once on the test set.
Compare with the default parameters and measure how much the optimization is worth.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from scipy.stats import randint, uniform
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

df = ds.loans(n=5000)
df["ratio"] = df["amount"] / df["income"]
X = pd.concat([df[["ratio", "age", "income", "job_tenure", "amount"]], pd.get_dummies(df["contract_type"], dtype=float)], axis=1)
y = df["defaulted"]
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=0)
space = {"max_depth": randint(2, 7), "min_child_weight": randint(1, 30), "subsample": uniform(0.5, 0.5), "colsample_bytree": uniform(0.5, 0.5),
         "reg_lambda": [0.1, 1, 5, 20], "gamma": [0, 0.5, 2, 5]}
base = dict(learning_rate=0.05, eval_metric="auc", n_jobs=-1, random_state=0)

## Commented solution

In [ ]:
def with_early_stopping(**params):
    m = xgb.XGBClassifier(n_estimators=2000, early_stopping_rounds=30, **base, **params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return m


m0 = with_early_stopping()
default_auc = roc_auc_score(y_te, m0.predict_proba(X_te)[:, 1])
print(f"defaults with early stopping: {m0.best_iteration + 1} rounds | test AUC {default_auc:.4f}")

In [ ]:
search = RandomizedSearchCV(xgb.XGBClassifier(n_estimators=200, **base), space, n_iter=20, cv=StratifiedKFold(3, shuffle=True, random_state=0),
                            scoring="roc_auc", random_state=0, n_jobs=1).fit(X_tr, y_tr)
best = search.best_params_
cv_auc = search.best_score_
n_fits = len(search.cv_results_["params"]) * 3
print("best parameters:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in best.items()}, f"| CV AUC {cv_auc:.4f} | trainings: {n_fits}")

In [ ]:
m1 = with_early_stopping(**best)
tuned_auc = roc_auc_score(y_te, m1.predict_proba(X_te)[:, 1])
final_rounds = m1.best_iteration + 1
gain = tuned_auc - default_auc
within_noise = bool(abs(gain) < 0.02)
print(f"optimized: {final_rounds} rounds | test AUC {tuned_auc:.4f} | gain over the defaults: {gain:+.4f} | within the noise (less than 2 points): {within_noise}")

## Check

In [ ]:
with verify():
    check(0.7 < default_auc < 0.87, "test AUC with the defaults between 0.70 and 0.87", "XGBClassifier(learning_rate=0.05, ...) with early stopping")
    check(set(best) == set(space) and n_fits == 60 and 0.72 < cv_auc < 0.87, "the search returns the six parameters; 60 trainings; cross-validation AUC between 0.72 and 0.87", "RandomizedSearchCV(n_iter=20, cv=3)")
    check(best["max_depth"] <= 4 and best["min_child_weight"] >= 5, "the signal is weak and noisy: the search prefers shallow trees (depth 4 or less) and a high min_child_weight (5 or more)", "conservative hyperparameters")
    check(0.7 < tuned_auc < 0.87 and final_rounds >= 20, "test AUC of the optimized model between 0.70 and 0.87 with at least 20 rounds", "early stopping on the validation set")
    check(close(gain, tuned_auc - default_auc) and abs(gain) < 0.04 and within_noise is True, "the gain over the defaults is small (less than 4 points, within the noise)", "gain = tuned_auc - default_auc")
done()

## What you learned

- **A random search with 20 combinations does the job:** with 6 hyperparameters a grid would be huge, but 20 random points almost always find a good region (Bergstra and Bengio, M3 F2). It costs 60 trainings (20 × 3 folds), a few seconds with `n_estimators=200`.
- **The search confirms what the problem asks for:** with a weak, noisy signal (AUC around 0.80) it prefers **shallow**, conservative trees (`max_depth` 2-3, high `min_child_weight`): deep boosting would memorize the noise.
- **The gain on the test set is modest:** +0.019 of AUC (from 0.770 to 0.789). With 1000 test examples the uncertainty of an AUC is about a point and a half, so the advantage is at the edge of the noise. It is a common result: the XGBoost defaults + early stopping are already very close to the optimum. The order of priority: first early stopping and learning rate, then depth and `min_child_weight`, then the rest.
- **Order of the steps:** search the hyperparameters with `n_estimators` fixed and learning rate fixed, then fix the rounds with early stopping on a validation set, then evaluate on the test set. Mixing the two phases (early stopping inside the search, with a different validation set per fold) is possible but more expensive.

## Common pitfall

Reporting the cross-validation AUC of the best point of the search as the estimate of the performance: it is the **maximum** of 20 noisy estimates (selection optimism, M3 C3). The number to report is the one on the test set, touched once.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you set up the tuning of XGBoost in practice?</summary>

I start from sensible defaults (learning rate 0.05, max_depth 4, subsample and colsample 0.8), choose the rounds with early stopping on a validation set, then run a random search (20-50 combinations) over max_depth, min_child_weight, subsample, colsample_bytree, reg_lambda and gamma with cross-validation, and retrain with early stopping. I evaluate once on the test set.
I expect a modest gain over the defaults and compare it with the uncertainty of the estimate.

</details>